In [1]:
import subprocess
import sys

def install_if_missing(package):
    """Check if package is installed, if not install it."""
    try:
        __import__(package)
    except ImportError:
        subprocess.check_call([sys.executable, "-m", "pip", "install", package])

# Install packages
install_if_missing('pandas')
install_if_missing('numpy')
install_if_missing('matplotlib')
install_if_missing('pylab')
install_if_missing('IPython')
install_if_missing('astropy')
install_if_missing('scipy')
install_if_missing('plotly')
install_if_missing('kaleido')

# Now import them
import pandas as pd 
import numpy as np 
import matplotlib.pyplot as plt
from astropy.coordinates import SkyCoord, ICRS, Galactic, GeocentricMeanEcliptic
from astropy.time import Time
import astropy.units as u
from scipy.interpolate import interp1d
import kaleido

import time
import math

import pylab as py
from IPython.display import HTML

In [2]:
class Body:
    def __init__(self, mass, position, velocity):
        self.mass = mass
        self.position = position
        self.velocity = velocity

    def copy(self):
        return Body(self.mass, self.position[:], self.velocity[:])

In [3]:
#parameters
G = 6.673e-11 #N m^2 / kg^2
M_Earth  = 6.0e24
M_Sun = 1.989e30 #kg
M_Moon =  7.342e22 #kg
M_oculus = 500 #kg
M_lux = 500 #kg

R_earth = 6371e3 #m
R_orbit = 91e6+R_earth #m

d_spacecraft = R_orbit*0.0 #m
orientation_polar = 0    * math.pi / 180 # radians
orientation_azimuthal = 0 * math.pi / 180 # radians

v_spacecraft_0 = math.sqrt(G*M_Earth/R_orbit) # m/s
T_orbit = 2*math.pi*R_orbit/v_spacecraft_0
T_orbit_hours = T_orbit / 3600
T_orbit_days = T_orbit_hours / 24
T_year = 365.25 * 24 * 3600/12 # seconds

orbit_inclination = 51.6 * math.pi / 180 # radians

N_steps = 10000
dt = T_year / N_steps
t = np.linspace(0, T_year, N_steps)

# ─── J2 ─────────────────────────────────────────────────────
J2        = 1.08263e-3           # Earth J2 coefficient
mu        = G * M_Earth          # gravitational parameter [m^3/s^2]

# ─── SRP ─────────────────────────────────────────────────────
P_solar   = 4.56e-6              # N/m^2  solar radiation pressure at 1 AU
AU        = 149597870700      # m
C_R       = 1.5                  # reflectivity coefficient
A_sc      = 1.0                  # m^2  cross-sectional area
m_sc      = 500.0                # kg   (same as M_oculus / M_lux)

# ─── Epoch for ephemeris ─────────────────────────────────────
# Set this to your simulation start date/time
t0_epoch  = Time("2000-01-01T00:00:00", scale="utc")

In [4]:
def unit_xyz(coord):
    v = coord.cartesian.xyz.value
    return v / np.linalg.norm(v)

# North Galactic Pole in ICRS
ngp = SkyCoord(l=0*u.deg, b=90*u.deg, frame=Galactic()).transform_to(ICRS())
ngp_hat = unit_xyz(ngp)

# North Ecliptic Pole in mean geocentric ecliptic coordinates, then to ICRS
nep = SkyCoord(lon=0*u.deg, lat=90*u.deg,
               frame=GeocentricMeanEcliptic(equinox=t0_epoch)).transform_to(ICRS())
nep_hat = unit_xyz(nep)

angle = np.degrees(np.arccos(np.clip(np.dot(ngp_hat, nep_hat), -1.0, 1.0)))
print(angle)   # should be about 60.2 deg

60.19424737449161


In [5]:
import numpy as np
import astropy.units as u
from astropy.time import Time
from astropy.coordinates import (
    SkyCoord, ICRS, GCRS, Galactic, GeocentricMeanEcliptic,
    get_body_barycentric_posvel
)
from scipy.interpolate import interp1d

def _unit(v):
    v = np.asarray(v, dtype=float)
    return v / np.linalg.norm(v)

def _xyz_m(coord):
    return np.array([
        coord.cartesian.x.to_value(u.m),
        coord.cartesian.y.to_value(u.m),
        coord.cartesian.z.to_value(u.m),
    ])

def get_sun_moon_eci(t_sec):
    """
    Return Sun and Moon geocentric position vectors [m] in GCRS.
    """
    t = t0_epoch + t_sec * u.s

    sun_bary = get_body_barycentric_posvel("sun", t, ephemeris="builtin")[0]
    sun_icrs = SkyCoord(sun_bary, frame="icrs", obstime=t, representation_type="cartesian")
    sun_gcrs = sun_icrs.transform_to(GCRS(obstime=t))
    r_sun = _xyz_m(sun_gcrs)

    moon_bary = get_body_barycentric_posvel("moon", t, ephemeris="builtin")[0]
    moon_icrs = SkyCoord(moon_bary, frame="icrs", obstime=t, representation_type="cartesian")
    moon_gcrs = moon_icrs.transform_to(GCRS(obstime=t))
    r_moon = _xyz_m(moon_gcrs)

    return r_sun, r_moon

def get_galactic_center_gcrs(epoch):
    gc_icrs = SkyCoord(l=0*u.deg, b=0*u.deg, frame=Galactic()).transform_to(ICRS())
    gc_gcrs = gc_icrs.transform_to(GCRS(obstime=epoch))
    return _unit(gc_gcrs.cartesian.xyz.to_value(u.one))

def get_north_galactic_pole_gcrs(epoch):
    ngp_icrs = SkyCoord(l=0*u.deg, b=90*u.deg, frame=Galactic()).transform_to(ICRS())
    ngp_gcrs = ngp_icrs.transform_to(GCRS(obstime=epoch))
    return _unit(ngp_gcrs.cartesian.xyz.to_value(u.one))

def get_north_ecliptic_pole_gcrs(epoch):
    nep_icrs = SkyCoord(
        lon=0*u.deg,
        lat=90*u.deg,
        frame=GeocentricMeanEcliptic(equinox=epoch)
    ).transform_to(ICRS())
    nep_gcrs = nep_icrs.transform_to(GCRS(obstime=epoch))
    return _unit(nep_gcrs.cartesian.xyz.to_value(u.one))

print("Pre-computing Sun/Moon ephemeris table...")
sample_idx = np.arange(0, N_steps, 100)

_sun_samp = np.zeros((len(sample_idx), 3))
_moon_samp = np.zeros((len(sample_idx), 3))

for k, idx in enumerate(sample_idx):
    _sun_samp[k], _moon_samp[k] = get_sun_moon_eci(t[idx])

_sun_interp = interp1d(t[sample_idx], _sun_samp, axis=0, fill_value="extrapolate")
_moon_interp = interp1d(t[sample_idx], _moon_samp, axis=0, fill_value="extrapolate")

def r_sun_at(t_sec):
    return _sun_interp(t_sec)

def r_moon_at(t_sec):
    return _moon_interp(t_sec)

r_gc_hat = get_galactic_center_gcrs(t0_epoch)
r_ngp_hat = get_north_galactic_pole_gcrs(t0_epoch)
r_nep_hat = get_north_ecliptic_pole_gcrs(t0_epoch)

angle_ngp_nep = np.degrees(np.arccos(np.clip(np.dot(r_ngp_hat, r_nep_hat), -1.0, 1.0)))

print("Ephemeris table ready.")
print(f"Galactic Center GCRS unit vector: {r_gc_hat}")
print(f"North Galactic Pole GCRS unit vector: {r_ngp_hat}")
print(f"North Ecliptic Pole GCRS unit vector: {r_nep_hat}")
print(f"NGP–NEP angle = {angle_ngp_nep:.3f} deg")

Pre-computing Sun/Moon ephemeris table...
Ephemeris table ready.
Galactic Center GCRS unit vector: [-0.05497406 -0.8734331  -0.48383103]
North Galactic Pole GCRS unit vector: [-0.86769072 -0.1980752   0.45593752]
North Ecliptic Pole GCRS unit vector: [-2.53993554e-07 -3.97776203e-01  9.17482475e-01]
NGP–NEP angle = 60.191 deg


In [7]:
import plotly.graph_objects as go
import numpy as np

# --- Data Preparation (using your existing functions) ---
sun_positions = r_sun_at(t)
moon_positions = r_moon_at(t)

# Translate to Sun-centered frame
earth_sc = -sun_positions
moon_sc = moon_positions - sun_positions
gc_vec = r_gc_hat * 1e11 # Scale for visualization
ngp_vec = r_ngp_hat * 1e11  # North Galactic Pole
nep_vec = r_nep_hat * 1e11  # North Ecliptic Pole

# --- Plotly Figure ---
fig = go.Figure()

# Add Earth
fig.add_trace(go.Scatter3d(x=earth_sc[:,0], y=earth_sc[:,1], z=earth_sc[:,2],
    mode='lines', name='Earth Orbit', line=dict(color='blue', width=4)))

# Add Moon
fig.add_trace(go.Scatter3d(x=moon_sc[:,0], y=moon_sc[:,1], z=moon_sc[:,2],
    mode='lines', name='Moon Orbit', line=dict(color='gray', width=2)))

# Add Sun
fig.add_trace(go.Scatter3d(x=[0], y=[0], z=[0],
    mode='markers', name='Sun', marker=dict(size=10, color='orange')))

# Add Galactic Center direction
fig.add_trace(go.Scatter3d(x=[0, gc_vec[0]], y=[0, gc_vec[1]], z=[0, gc_vec[2]],
    mode='lines', name='Galactic Center', line=dict(color='red', width=6)))

# Add North Galactic Pole direction
fig.add_trace(go.Scatter3d(x=[0, ngp_vec[0]], y=[0, ngp_vec[1]], z=[0, ngp_vec[2]],
    mode='lines', name='North Galactic Pole', line=dict(color='green', width=6)))

# Add North Ecliptic Pole direction
fig.add_trace(go.Scatter3d(x=[0, nep_vec[0]], y=[0, nep_vec[1]], z=[0, nep_vec[2]],
    mode='lines', name='North Ecliptic Pole', line=dict(color='purple', width=6)))

# --- Layout ---
fig.update_layout(
    scene=dict(xaxis_title='X (m)', yaxis_title='Y (m)', zaxis_title='Z (m)', aspectmode='data'),
    title="Sun-Centered Orbits",
    margin=dict(l=0, r=0, b=0, t=40)
)

# Show interactively in VS Code
fig.show()

# Save to an HTML file to open in any browser
fig.write_html("interactive_orbits.html")
print("Interactive plot generated and saved as 'interactive_orbits.html'")

ValueError: Mime type rendering requires nbformat>=4.2.0 but it is not installed

In [6]:
def compute_perturbed_acceleration(position, t_sec=0.0):
    """
    Full perturbed acceleration [m/s^2] in ECI frame.

    Perturbations:
        - J2 oblateness
        - Sun third-body  (live astropy ephemeris)
        - Moon third-body (live astropy ephemeris)
        - Solar radiation pressure
    """
    r_sun  = r_sun_at(t_sec)
    r_moon = r_moon_at(t_sec)

    r_norm = np.linalg.norm(position)

    # ── 1. Two-body central ────────────────────────────────────────────────
    a_2body = -mu / r_norm**3 * position

    # ── 2. J2 ─────────────────────────────────────────────────────────────
    z2_r2  = (position[2] / r_norm) ** 2
    fJ2    = 1.5 * J2 * mu * R_earth**2 / r_norm**5
    a_J2   = fJ2 * np.array([
        position[0] * (5.0 * z2_r2 - 1.0),
        position[1] * (5.0 * z2_r2 - 1.0),
        position[2] * (5.0 * z2_r2 - 3.0)
    ])

    # ── 3. Sun third-body ─────────────────────────────────────────────────
    d_sun    = r_sun - position
    a_sun    = G * M_Sun  * (d_sun  / np.linalg.norm(d_sun)**3
                             - r_sun  / np.linalg.norm(r_sun)**3)

    # ── 4. Moon third-body ────────────────────────────────────────────────
    d_moon   = r_moon - position
    a_moon   = G * M_Moon * (d_moon / np.linalg.norm(d_moon)**3
                             - r_moon / np.linalg.norm(r_moon)**3)

    # ── 5. Solar Radiation Pressure ───────────────────────────────────────
    sc_from_sun      = position - r_sun
    sc_from_sun_norm = np.linalg.norm(sc_from_sun)
    sc_from_sun_hat  = sc_from_sun / sc_from_sun_norm
    P_scaled         = P_solar * (AU / sc_from_sun_norm) ** 2
    a_srp            = (C_R * P_scaled * A_sc / m_sc) * sc_from_sun_hat

    return a_2body + a_J2 + a_sun + a_moon + a_srp, d_sun, d_moon


In [7]:
def compute_gravitational_acceleration(position, time_sec=0.0):
    r = np.linalg.norm(position)
    return -G * M_Earth / r**3 * position

In [8]:
def make_initial_conditions(polar_angle, inclination):
    r_oculus_0 = np.array([
        R_orbit * np.cos(inclination),
        0.0,
        R_orbit * np.sin(inclination)
    ])

    v_oculus_0 = np.array([
        0.0,
        v_spacecraft_0,
        0.0
    ])

    r_lux_0 = r_oculus_0 + np.array([
        d_spacecraft * np.cos(polar_angle) * np.cos(orientation_azimuthal),
        d_spacecraft * np.cos(polar_angle) * np.sin(orientation_azimuthal),
        d_spacecraft * np.sin(polar_angle)
    ])

    v_lux_0 = v_oculus_0.copy()

    orbital_normal_vector = np.array([
        -np.sin(inclination),
        0.0,
        np.cos(inclination)
    ])

    return r_oculus_0, v_oculus_0, r_lux_0, v_lux_0, orbital_normal_vector


In [9]:
class Simulation:
    def __init__(self, body1, body2, dt, nsteps, gravity_func):
        self.body1        = body1
        self.body2        = body2
        self.dt           = dt
        self.nsteps       = nsteps
        self.gravity_func = gravity_func
        self.sun_relative_temp = 0
        self.moon_relative_temp = 0

        self.relative_offset = self.body2.position.copy() - self.body1.position.copy()

        self.r1 = np.zeros((nsteps, 3))
        self.v1 = np.zeros((nsteps, 3))
        self.r2 = np.zeros((nsteps, 3))
        self.v2 = np.zeros((nsteps, 3))
        self.f2_constraint = np.zeros((nsteps, 3))
        self.sun_relative = np.zeros((nsteps, 3))
        self.moon_relative = np.zeros((nsteps, 3))

        self.total_impulse = 0.0
        self.has_run       = False
        self._store_step(0)

    def _store_step(self, i):
        self.r1[i] = self.body1.position
        self.v1[i] = self.body1.velocity
        self.r2[i] = self.body2.position
        self.v2[i] = self.body2.velocity

    def _rk4_step_with_constraint(self, step):
        dt = self.dt
        d0 = self.relative_offset

        r1_0 = self.body1.position.copy()
        v1_0 = self.body1.velocity.copy()

        def acc(r_vec):
            a, d_sun_temp, d_moon_temp = self.gravity_func(r_vec, step*dt)
            return a
        # ---------- stage 1 ----------
        r1_s1 = r1_0
        v1_s1 = v1_0
        a1_s1 = acc(r1_s1)

        r2_s1 = r1_s1 + d0
        v2_s1 = v1_s1.copy()
        a2_grav_s1 = acc(r2_s1)
        a2_req_s1 = a1_s1.copy()
        a2_con_s1 = a2_req_s1 - a2_grav_s1

        k1_r = v1_s1
        k1_v = a1_s1

        # ---------- stage 2 ----------
        r1_s2 = r1_0 + 0.5 * dt * k1_r
        v1_s2 = v1_0 + 0.5 * dt * k1_v
        a1_s2 = acc(r1_s2)

        r2_s2 = r1_s2 + d0
        v2_s2 = v1_s2.copy()
        a2_grav_s2 = acc(r2_s2)
        a2_req_s2 = a1_s2.copy()
        a2_con_s2 = a2_req_s2 - a2_grav_s2

        k2_r = v1_s2
        k2_v = a1_s2

        # ---------- stage 3 ----------
        r1_s3 = r1_0 + 0.5 * dt * k2_r
        v1_s3 = v1_0 + 0.5 * dt * k2_v
        a1_s3 = acc(r1_s3)

        r2_s3 = r1_s3 + d0
        v2_s3 = v1_s3.copy()
        a2_grav_s3 = acc(r2_s3)
        a2_req_s3 = a1_s3.copy()
        a2_con_s3 = a2_req_s3 - a2_grav_s3

        k3_r = v1_s3
        k3_v = a1_s3

        # ---------- stage 4 ----------
        r1_s4 = r1_0 + dt * k3_r
        v1_s4 = v1_0 + dt * k3_v
        a1_s4 = acc(r1_s4)

        r2_s4 = r1_s4 + d0
        v2_s4 = v1_s4.copy()
        a2_grav_s4 = acc(r2_s4)
        a2_req_s4 = a1_s4.copy()
        a2_con_s4 = a2_req_s4 - a2_grav_s4

        k4_r = v1_s4
        k4_v = a1_s4

        # ---------- final RK4 update ----------
        r1_new = r1_0 + dt * (k1_r + 2*k2_r + 2*k3_r + k4_r) / 6.0
        v1_new = v1_0 + dt * (k1_v + 2*k2_v + 2*k3_v + k4_v) / 6.0

        a1_rk4 = (a1_s1 + 2*a1_s2 + 2*a1_s3 + a1_s4) / 6.0
        a2_total_rk4 = (a2_req_s1 + 2*a2_req_s2 + 2*a2_req_s3 + a2_req_s4) / 6.0
        a2_grav_rk4 = (a2_grav_s1 + 2*a2_grav_s2 + 2*a2_grav_s3 + a2_grav_s4) / 6.0
        a2_con_rk4 = (a2_con_s1 + 2*a2_con_s2 + 2*a2_con_s3 + a2_con_s4) / 6.0

        r2_new = r1_new + d0
        v2_new = v1_new.copy()

        self.body1.position = r1_new
        self.body1.velocity = v1_new
        self.body2.position = r2_new
        self.body2.velocity = v2_new

        return a1_rk4, a2_total_rk4, a2_grav_rk4, a2_con_rk4

    def run(self, trim_start=0):
        for i in range(1, self.nsteps):
            a1_rk4, a2_total_rk4, a2_grav_rk4, a2_con_rk4 = self._rk4_step_with_constraint(i)
            self._store_step(i)

            self.f2_constraint[i] = self.body2.mass * a2_con_rk4
            self.sun_relative[i] = self.sun_relative_temp
            self.moon_relative[i] = self.moon_relative_temp
            self.total_impulse   += np.linalg.norm(self.f2_constraint[i]) * self.dt

        if trim_start > 0:
            self.r1 = self.r1[trim_start:]
            self.v1 = self.v1[trim_start:]
            self.r2 = self.r2[trim_start:]
            self.v2 = self.v2[trim_start:]
            self.f2_constraint = self.f2_constraint[trim_start:]

        self.has_run = True

    # ── plotting / CSV methods unchanged ──────────────────────────────────
    
    def plot_trajectories(self):
        fig = plt.figure(figsize=(10, 10))
        ax = fig.add_subplot(111, projection="3d")

        ax.plot(self.r1[:, 0], self.r1[:, 1], self.r1[:, 2], label="Oculus")
        ax.plot(self.r2[:, 0], self.r2[:, 1], self.r2[:, 2], label="Lux")
        ax.scatter(0, 0, 0, color="gold", s=100, label="Earth")

        ax.set_xlabel("X Position (m)")
        ax.set_ylabel("Y Position (m)")
        ax.set_zlabel("Z Position (m)")
        ax.set_title("3D Trajectories")
        ax.legend()
        plt.show()

    def plot_solar_system(self):
        if not self.has_run:
            raise RuntimeError("Run simulation before plotting solar system")

        fig = plt.figure(figsize=(10, 10))
        ax = fig.add_subplot(111, projection="3d")

        # Earth at origin
        ax.scatter(0, 0, 0, color="blue", s=100, label="Earth")

        # Sun and Moon trajectories in Earth-centered coordinates
        ax.plot(
            self.sun_relative[:, 0],
            self.sun_relative[:, 1],
            self.sun_relative[:, 2],
            color="orange",
            label="Sun",
        )
        ax.plot(
            self.moon_relative[:, 0],
            self.moon_relative[:, 1],
            self.moon_relative[:, 2],
            color="gray",
            label="Moon",
        )

        ax.set_xlabel("X Position (m)")
        ax.set_ylabel("Y Position (m)")
        ax.set_zlabel("Z Position (m)")
        ax.set_title("Solar System Bodies Trajectories (Earth-centered)")
        ax.legend()
        plt.show()
    

    def plot_constraint_force(self):
        plt.figure(figsize=(10, 5))
        plt.plot(np.linalg.norm(self.f2_constraint, axis=1))
        plt.xlabel("Time Step")
        plt.ylabel("Constraint Force (N)")
        plt.title("Constraint Force Magnitude on Body 2")
        plt.grid(True)
        plt.show()
    
    def print_total_impulse(self):
        print(f"Total Impulse Exerted on Body 2: {self.total_impulse:.2e} N·s")
    
    def save_to_csv(self, filename):
        data = {
            "time": np.arange(len(self.r1)) * self.dt,
            "r1_x": self.r1[:, 0],
            "r1_y": self.r1[:, 1],
            "r1_z": self.r1[:, 2],
            "v1_x": self.v1[:, 0],
            "v1_y": self.v1[:, 1],
            "v1_z": self.v1[:, 2],
            "r2_x": self.r2[:, 0],
            "r2_y": self.r2[:, 1],
            "r2_z": self.r2[:, 2],
            "v2_x": self.v2[:, 0],
            "v2_y": self.v2[:, 1],
            "v2_z": self.v2[:, 2],
            "f2_constraint_x": self.f2_constraint[:, 0],
            "f2_constraint_y": self.f2_constraint[:, 1],
            "f2_constraint_z": self.f2_constraint[:, 2]
        }
        df = pd.DataFrame(data)
        df.to_csv(filename, index=False)



In [10]:

# --- Updated formation-facing-galactic-center setup + 1-year animated heliocentric visualization ---
import plotly.graph_objects as go

# Keep body configuration and RK4 engine; only replace IC setup and add animation helper.

R_orbit = 100*10**6
d_spacecraft = R_orbit*0.1 # m, nonzero formation baseline for visualization and constraint analysis
N_steps = 30000
T_year = 365.25 * 24 * 3600

dt = T_year / N_steps
t = np.linspace(0, T_year, N_steps)

sample_idx = np.arange(0, N_steps, 100)
_sun_samp = np.zeros((len(sample_idx), 3))
_moon_samp = np.zeros((len(sample_idx), 3))
for k, idx in enumerate(sample_idx):
    _sun_samp[k], _moon_samp[k] = get_sun_moon_eci(t[idx])
_sun_interp = interp1d(t[sample_idx], _sun_samp, axis=0, fill_value='extrapolate')
_moon_interp = interp1d(t[sample_idx], _moon_samp, axis=0, fill_value='extrapolate')

def r_sun_at(t_sec):
    return _sun_interp(t_sec)

def r_moon_at(t_sec):
    return _moon_interp(t_sec)

r_gc_hat = get_galactic_center_gcrs(t0_epoch)
r_ngp_hat = get_north_galactic_pole_gcrs(t0_epoch)
r_nep_hat = get_north_ecliptic_pole_gcrs(t0_epoch)

def make_initial_conditions_face_gc(inclination, separation=d_spacecraft):
    r_oculus_0 = np.array([
        R_orbit * np.cos(inclination),
        0.0,
        R_orbit * np.sin(inclination)
    ], dtype=float)

    v_oculus_0 = np.array([0.0, v_spacecraft_0, 0.0], dtype=float)

    gc_hat = r_gc_hat / np.linalg.norm(r_gc_hat)

    rel_offset = separation * gc_hat
    r_lux_0 = r_oculus_0 + rel_offset
    v_lux_0 = v_oculus_0.copy()

    polar = np.arcsin(np.clip(gc_hat[2], -1.0, 1.0))
    azim = np.arctan2(gc_hat[1], gc_hat[0])

    orbital_normal_vector = np.cross(r_oculus_0, v_oculus_0)
    orbital_normal_vector = orbital_normal_vector / np.linalg.norm(orbital_normal_vector)

    return (
        r_oculus_0,
        v_oculus_0,
        r_lux_0,
        v_lux_0,
        orbital_normal_vector,
        polar,
        azim,
        rel_offset
    )

r_oculus_0, v_oculus_0, r_lux_0, v_lux_0, orbital_normal_vector, orientation_polar_gc, orientation_azimuthal_gc, rel_offset_gc = make_initial_conditions_face_gc(21)

sim_year = Simulation(
    Body(M_oculus, r_oculus_0.copy(), v_oculus_0.copy()),
    Body(M_lux, r_lux_0.copy(), v_lux_0.copy()),
    dt,
    N_steps,
    compute_perturbed_acceleration
)
sim_year.run(trim_start=0)

sim_year.sun_relative = np.array([r_sun_at(tt) for tt in t])

print(f'GC-facing initial offset [m]: {rel_offset_gc}')
print(f'Polar angle [deg]: {np.degrees(orientation_polar_gc):.3f}')
print(f'Azimuth angle [deg]: {np.degrees(orientation_azimuthal_gc):.3f}')
print(f'Initial offset aligned with GC cosine: {np.dot(rel_offset_gc/np.linalg.norm(rel_offset_gc), r_gc_hat):.6f}')


GC-facing initial offset [m]: [ -549740.59510483 -8734331.0163167  -4838310.34303319]
Polar angle [deg]: -28.936
Azimuth angle [deg]: -93.601
Initial offset aligned with GC cosine: 1.000000


In [19]:

# --- Reliable static plot + GIF export ---
import matplotlib.pyplot as plt
from matplotlib.animation import FuncAnimation, PillowWriter
from mpl_toolkits.mplot3d import Axes3D

sun_geo = sim_year.sun_relative.copy()
earth_sc = (-sun_geo).copy()
oculus_sc = (sim_year.r1 - sun_geo).copy()
lux_sc = (sim_year.r2 - sun_geo).copy()

def add_3d_arrow(fig, start, end, color='red', name='Arrow',
                 shaft_width=6, head_frac=0.10, head_scale=0.08,
                 showlegend=True):
    start = np.asarray(start, dtype=float)
    end = np.asarray(end, dtype=float)

    vec = end - start
    L = np.linalg.norm(vec)
    if L == 0:
        return

    vhat = vec / L
    head_len = head_frac * L
    shaft_end = end - head_len * vhat

    fig.add_trace(go.Scatter3d(
        x=[start[0], shaft_end[0]],
        y=[start[1], shaft_end[1]],
        z=[start[2], shaft_end[2]],
        mode='lines',
        line=dict(color=color, width=shaft_width),
        name=name,
        showlegend=showlegend,
        hovertemplate=f'{name}<extra></extra>'
    ))

    fig.add_trace(go.Cone(
        x=[end[0]],
        y=[end[1]],
        z=[end[2]],
        u=[head_len * vhat[0]],
        v=[head_len * vhat[1]],
        w=[head_len * vhat[2]],
        anchor='tip',
        sizemode='absolute',
        sizeref=head_scale * L,
        colorscale=[[0, color], [1, color]],
        showscale=False,
        hoverinfo='skip',
        showlegend=False
    ))

fig = go.Figure()

# Earth full orbit
fig.add_trace(go.Scatter3d(
    x=earth_sc[:, 0],
    y=earth_sc[:, 1],
    z=earth_sc[:, 2],
    mode='lines',
    name='Earth orbit',
    line=dict(color='royalblue', width=5)
))

# Earth start marker
fig.add_trace(go.Scatter3d(
    x=[earth_sc[0, 0]],
    y=[earth_sc[0, 1]],
    z=[earth_sc[0, 2]],
    mode='markers',
    name='Earth start',
    marker=dict(size=5, color='royalblue')
))

# Oculus full trajectory
fig.add_trace(go.Scatter3d(
    x=oculus_sc[:, 0],
    y=oculus_sc[:, 1],
    z=oculus_sc[:, 2],
    mode='lines',
    name='Oculus',
    line=dict(color='cyan', width=3)
))

# Oculus start marker
fig.add_trace(go.Scatter3d(
    x=[oculus_sc[0, 0]],
    y=[oculus_sc[0, 1]],
    z=[oculus_sc[0, 2]],
    mode='markers',
    name='Oculus start',
    marker=dict(size=4, color='cyan')
))

# Lux full trajectory
fig.add_trace(go.Scatter3d(
    x=lux_sc[:, 0],
    y=lux_sc[:, 1],
    z=lux_sc[:, 2],
    mode='lines',
    name='Lux',
    line=dict(color='magenta', width=3)
))

# Lux start marker
fig.add_trace(go.Scatter3d(
    x=[lux_sc[0, 0]],
    y=[lux_sc[0, 1]],
    z=[lux_sc[0, 2]],
    mode='markers',
    name='Lux start',
    marker=dict(size=4, color='magenta')
))

# Sun at origin
fig.add_trace(go.Scatter3d(
    x=[0],
    y=[0],
    z=[0],
    mode='markers',
    name='Sun',
    marker=dict(size=10, color='orange')
))

# Galactic Center arrow
add_3d_arrow(fig, [0, 0, 0], r_gc_hat * AU, color='red', name='Galactic Center')

fig.update_layout(
    scene=dict(
        aspectmode='data',
        xaxis=dict(title='X (m)'),
        yaxis=dict(title='Y (m)'),
        zaxis=dict(title='Z (m)')
    ),
    margin=dict(l=0, r=0, b=0, t=50)
)

#fig.show()
fig.write_image(
    "galactic_frame_static.pdf",
    width=1100,
    height=700,
)
fig_sun_centered = go.Figure(fig)

In [31]:
import plotly.graph_objects as go
import numpy as np

# Number of steps corresponding to one orbital period
steps_one_orbit = int(N_steps * T_orbit / T_year)

gc_scale = R_orbit * 1.5

fig = go.Figure()

# Earth at origin
fig.add_trace(go.Scatter3d(
    x=[0], y=[0], z=[0],
    mode='markers',
    name='Earth',
    marker=dict(size=8, color='royalblue')
))

# Oculus — one orbit only
fig.add_trace(go.Scatter3d(
    x=sim_year.r1[:steps_one_orbit, 0],
    y=sim_year.r1[:steps_one_orbit, 1],
    z=sim_year.r1[:steps_one_orbit, 2],
    mode='lines',
    name='Oculus',
    line=dict(color='cyan', width=3)
))

# Lux — one orbit only
fig.add_trace(go.Scatter3d(
    x=sim_year.r2[:steps_one_orbit, 0],
    y=sim_year.r2[:steps_one_orbit, 1],
    z=sim_year.r2[:steps_one_orbit, 2],
    mode='lines',
    name='Lux',
    line=dict(color='magenta', width=3)
))

# Galactic Center arrow (shaft)
gc_end = r_gc_hat * gc_scale
shaft_end = gc_end * 0.88
fig.add_trace(go.Scatter3d(
    x=[0, shaft_end[0]],
    y=[0, shaft_end[1]],
    z=[0, shaft_end[2]],
    mode='lines',
    name='Galactic Center',
    line=dict(color='red', width=5),
    hovertemplate='Galactic Center<extra></extra>'
))

# Galactic Center arrow cone
head_len = gc_scale * 0.12
fig.add_trace(go.Cone(
    x=[gc_end[0]], y=[gc_end[1]], z=[gc_end[2]],
    u=[head_len * r_gc_hat[0]],
    v=[head_len * r_gc_hat[1]],
    w=[head_len * r_gc_hat[2]],
    anchor='tip',
    sizemode='absolute',
    sizeref=gc_scale * 0.08,
    colorscale=[[0, 'red'], [1, 'red']],
    showscale=False,
    hoverinfo='skip',
    showlegend=False
))

fig.update_layout(
    #title='Earth-centered view: One orbit — Oculus & Lux with Galactic Center direction',
    scene=dict(
        aspectmode='data',
        xaxis=dict(title='X (m)'),
        yaxis=dict(title='Y (m)'),
        zaxis=dict(title='Z (m)')
    ),
    scene_camera=dict(
        eye=dict(x=2.45*2/3, y=1.30*2/3, z=0.95*2/3)
    ),
    margin=dict(l=0, r=0, b=0, t=50),
)

fig.write_image(
    "gc-orbits.pdf",
    width=1100,
    height=700,
)
fig_static = go.Figure(fig)


In [ ]:
# ---------- GIF animation ----------
frame_count = 180
frame_idx = np.linspace(0, len(earth_sc) - 1, frame_count, dtype=int)

fig_gif = plt.figure(figsize=(10, 8))
axg = fig_gif.add_subplot(111, projection='3d')
all_x = np.concatenate([earth_sc[:, 0], oculus_sc[:, 0], lux_sc[:, 0]])
all_y = np.concatenate([earth_sc[:, 1], oculus_sc[:, 1], lux_sc[:, 1]])
all_z = np.concatenate([earth_sc[:, 2], oculus_sc[:, 2], lux_sc[:, 2]])

max_range = np.array([all_x.max()-all_x.min(), all_y.max()-all_y.min(), all_z.max()-all_z.min()]).max() / 2.0
mid_x = (all_x.max()+all_x.min()) * 0.5
mid_y = (all_y.max()+all_y.min()) * 0.5
mid_z = (all_z.max()+all_z.min()) * 0.5

axg.set_xlim(mid_x - max_range, mid_x + max_range)
axg.set_ylim(mid_y - max_range, mid_y + max_range)
axg.set_zlim(mid_z - max_range, mid_z + max_range)

arrow_len = max_range * 0.8
axg.set_title('Sun-centered animation: Earth and GC-aligned formation')
axg.set_xlabel('X (m)')
axg.set_ylabel('Y (m)')
axg.set_zlabel('Z (m)')
axg.view_init(elev=20, azim=35)

sun_scatter = axg.scatter([0], [0], [0], color='orange', s=90, label='Sun')
axg.quiver(0, 0, 0, *(r_gc_hat * arrow_len), color='red', linewidth=2.2)

earth_line, = axg.plot([], [], [], color='royalblue', lw=1.8, label='Earth orbit')
earth_point, = axg.plot([], [], [], marker='o', color='royalblue', ms=6, linestyle='None')
oculus_line, = axg.plot([], [], [], color='cyan', lw=1.2, label='Oculus')
oculus_point, = axg.plot([], [], [], marker='o', color='cyan', ms=4, linestyle='None')
lux_line, = axg.plot([], [], [], color='magenta', lw=1.2, label='Lux')
lux_point, = axg.plot([], [], [], marker='o', color='magenta', ms=4, linestyle='None')
axg.legend(loc='upper left')

def update(frame_number):
    i = frame_idx[frame_number]

    earth_line.set_data(earth_sc[:i+1,0], earth_sc[:i+1,1])
    earth_line.set_3d_properties(earth_sc[:i+1,2])
    earth_point.set_data([earth_sc[i,0]], [earth_sc[i,1]])
    earth_point.set_3d_properties([earth_sc[i,2]])

    oculus_line.set_data(oculus_sc[:i+1,0], oculus_sc[:i+1,1])
    oculus_line.set_3d_properties(oculus_sc[:i+1,2])
    oculus_point.set_data([oculus_sc[i,0]], [oculus_sc[i,1]])
    oculus_point.set_3d_properties([oculus_sc[i,2]])

    lux_line.set_data(lux_sc[:i+1,0], lux_sc[:i+1,1])
    lux_line.set_3d_properties(lux_sc[:i+1,2])
    lux_point.set_data([lux_sc[i,0]], [lux_sc[i,1]])
    lux_point.set_3d_properties([lux_sc[i,2]])

    return earth_line, earth_point, oculus_line, oculus_point, lux_line, lux_point

anim = FuncAnimation(fig_gif, update, frames=len(frame_idx), interval=60, blit=False)
anim.save('orbits_animation_year.gif', writer=PillowWriter(fps=15))
plt.close(fig_gif)

In [15]:
import plotly.graph_objects as go
import numpy as np

steps_one_orbit = int(N_steps * T_orbit / T_year)
gc_scale = R_orbit * 1.5
head_len = gc_scale * 0.12

n_frames = 500
frame_idx = np.linspace(0, N_steps - 1, n_frames, dtype=int)

sun_rel = sim_year.sun_relative

def sun_arrow_traces(i):
    sun_dir = sun_rel[i]
    sun_hat = sun_dir / np.linalg.norm(sun_dir)
    arrow_base = sun_hat * gc_scale
    arrow_tip  = sun_hat * (gc_scale * 0.15)
    cone_dir   = arrow_tip - arrow_base
    cone_hat   = cone_dir / np.linalg.norm(cone_dir)

    shaft = go.Scatter3d(
        x=[arrow_base[0], arrow_tip[0]],
        y=[arrow_base[1], arrow_tip[1]],
        z=[arrow_base[2], arrow_tip[2]],
        mode='lines', line=dict(color='orange', width=5),
        showlegend=False, hovertemplate='Sun direction<extra></extra>'
    )
    cone = go.Cone(
        x=[arrow_tip[0]], y=[arrow_tip[1]], z=[arrow_tip[2]],
        u=[head_len * cone_hat[0]],
        v=[head_len * cone_hat[1]],
        w=[head_len * cone_hat[2]],
        anchor='tip', sizemode='absolute', sizeref=gc_scale * 0.08,
        colorscale=[[0, 'orange'], [1, 'orange']],
        showscale=False, hoverinfo='skip', showlegend=False
    )
    return shaft, cone

# ── Axis limits ────────────────────────────────────────────────────────────────

all_x = np.concatenate([sim_year.r1[:, 0], sim_year.r2[:, 0], [gc_end[0]], [0]])
all_y = np.concatenate([sim_year.r1[:, 1], sim_year.r2[:, 1], [gc_end[1]], [0]])
all_z = np.concatenate([sim_year.r1[:, 2], sim_year.r2[:, 2], [gc_end[2]], [0]])

pad = 0.1
x_range = [all_x.min() - pad * abs(all_x.min()), all_x.max() + pad * abs(all_x.max())]
y_range = [all_y.min() - pad * abs(all_y.min()), all_y.max() + pad * abs(all_y.max())]
z_range = [all_z.min() - pad * abs(all_z.min()), all_z.max() + pad * abs(all_z.max())]

# ── Base figure ────────────────────────────────────────────────────────────────

fig = go.Figure()

# Trace 0 — Earth
fig.add_trace(go.Scatter3d(
    x=[0], y=[0], z=[0],
    mode='markers', name='Earth',
    marker=dict(size=8, color='royalblue')
))

# Trace 1 — Oculus static trail
fig.add_trace(go.Scatter3d(
    x=sim_year.r1[:steps_one_orbit, 0],
    y=sim_year.r1[:steps_one_orbit, 1],
    z=sim_year.r1[:steps_one_orbit, 2],
    mode='lines', name='Oculus trail',
    line=dict(color='cyan', width=2, dash='dot')
))

# Trace 2 — Lux static trail
fig.add_trace(go.Scatter3d(
    x=sim_year.r2[:steps_one_orbit, 0],
    y=sim_year.r2[:steps_one_orbit, 1],
    z=sim_year.r2[:steps_one_orbit, 2],
    mode='lines', name='Lux trail',
    line=dict(color='magenta', width=2, dash='dot')
))

# Trace 3 — GC shaft
gc_end = r_gc_hat * gc_scale
fig.add_trace(go.Scatter3d(
    x=[0, gc_end[0] * 0.88],
    y=[0, gc_end[1] * 0.88],
    z=[0, gc_end[2] * 0.88],
    mode='lines', name='Galactic Center',
    line=dict(color='red', width=5),
    hovertemplate='Galactic Center<extra></extra>'
))

# Trace 4 — GC cone
fig.add_trace(go.Cone(
    x=[gc_end[0]], y=[gc_end[1]], z=[gc_end[2]],
    u=[head_len * r_gc_hat[0]],
    v=[head_len * r_gc_hat[1]],
    w=[head_len * r_gc_hat[2]],
    anchor='tip', sizemode='absolute', sizeref=gc_scale * 0.08,
    colorscale=[[0, 'red'], [1, 'red']],
    showscale=False, hoverinfo='skip', showlegend=False
))

# Trace 5 — Oculus marker (animated)
fig.add_trace(go.Scatter3d(
    x=[sim_year.r1[frame_idx[0], 0]],
    y=[sim_year.r1[frame_idx[0], 1]],
    z=[sim_year.r1[frame_idx[0], 2]],
    mode='markers', name='Oculus',
    marker=dict(size=5, color='cyan')
))

# Trace 6 — Lux marker (animated)
fig.add_trace(go.Scatter3d(
    x=[sim_year.r2[frame_idx[0], 0]],
    y=[sim_year.r2[frame_idx[0], 1]],
    z=[sim_year.r2[frame_idx[0], 2]],
    mode='markers', name='Lux',
    marker=dict(size=5, color='magenta')
))

# Traces 7 & 8 — Sun arrow (animated)
init_shaft, init_cone = sun_arrow_traces(frame_idx[0])

fig.add_trace(go.Scatter3d(
    x=init_shaft.x, y=init_shaft.y, z=init_shaft.z,
    mode='lines', name='Sun direction',
    line=dict(color='orange', width=5),
    hovertemplate='Sun direction<extra></extra>'
))

fig.add_trace(go.Cone(
    x=init_cone.x, y=init_cone.y, z=init_cone.z,
    u=init_cone.u, v=init_cone.v, w=init_cone.w,
    anchor='tip', sizemode='absolute', sizeref=gc_scale * 0.08,
    colorscale=[[0, 'orange'], [1, 'orange']],
    showscale=False, hoverinfo='skip', showlegend=False
))

# ── Frames — traces 5, 6, 7, 8 update each frame ──────────────────────────────

frames = []
for k, i in enumerate(frame_idx):
    shaft, cone = sun_arrow_traces(i)
    frames.append(go.Frame(
        data=[
            # Oculus marker
            go.Scatter3d(
                x=[sim_year.r1[i, 0]],
                y=[sim_year.r1[i, 1]],
                z=[sim_year.r1[i, 2]]
            ),
            # Lux marker
            go.Scatter3d(
                x=[sim_year.r2[i, 0]],
                y=[sim_year.r2[i, 1]],
                z=[sim_year.r2[i, 2]]
            ),
            # Sun shaft
            go.Scatter3d(x=shaft.x, y=shaft.y, z=shaft.z),
            # Sun cone
            go.Cone(
                x=cone.x, y=cone.y, z=cone.z,
                u=cone.u, v=cone.v, w=cone.w,
                anchor='tip', sizemode='absolute', sizeref=gc_scale * 0.08,
                colorscale=[[0, 'orange'], [1, 'orange']],
                showscale=False, hoverinfo='skip'
            )
        ],
        traces=[5, 6, 7, 8],
        name=f'frame_{k}'
    ))

fig.frames = frames

fig.update_layout(
    title='Earth-centered view: Full year — Animated spacecraft and Sun direction',
    scene=dict(
        aspectmode='manual',
        aspectratio=dict(x=1, y=1, z=1),
        xaxis=dict(title='X (m)', range=x_range),
        yaxis=dict(title='Y (m)', range=y_range),
        zaxis=dict(title='Z (m)', range=z_range)
    ),
    updatemenus=[dict(
        type='buttons', showactive=False,
        buttons=[
            dict(label='Play', method='animate',
                 args=[None, {'frame': {'duration': 80, 'redraw': True},
                              'fromcurrent': True,
                              'transition': {'duration': 0},
                              'mode': 'immediate'}]),
            dict(label='Pause', method='animate',
                 args=[[None], {'frame': {'duration': 0, 'redraw': False},
                                'transition': {'duration': 0},
                                'mode': 'immediate'}])
        ]
    )],
    margin=dict(l=0, r=0, b=0, t=50)
)

#fig.show()
fig.write_html('orbits_animated_full.html', include_plotlyjs=True, auto_play=False)
fig_animated = go.Figure(fig)

In [ ]:
import matplotlib.pyplot as plt
from matplotlib.animation import FuncAnimation, PillowWriter
from mpl_toolkits.mplot3d import Axes3D

n_frames = 120
frame_idx = np.linspace(0, N_steps - 1, n_frames, dtype=int)
sun_rel = sim_year.sun_relative

fig_gif = plt.figure(figsize=(10, 8))
ax = fig_gif.add_subplot(111, projection='3d')

# Static trails
ax.plot(sim_year.r1[:, 0], sim_year.r1[:, 1], sim_year.r1[:, 2],
        color='cyan', lw=1.5, ls='--', alpha=0.4, label='Oculus trail')
ax.plot(sim_year.r2[:, 0], sim_year.r2[:, 1], sim_year.r2[:, 2],
        color='magenta', lw=1.5, ls='--', alpha=0.4, label='Lux trail')
ax.scatter([0], [0], [0], color='royalblue', s=80, label='Earth')

# Static GC arrow
gc_end = r_gc_hat * gc_scale
ax.quiver(0, 0, 0, gc_end[0], gc_end[1], gc_end[2],
          color='red', linewidth=2, label='Galactic Center')

# Animated elements
oculus_pt, = ax.plot([], [], [], 'o', color='cyan', ms=6)
lux_pt,    = ax.plot([], [], [], 'o', color='magenta', ms=6)
sun_arrow  = ax.quiver(0, 0, 0, 1, 0, 0, color='orange', linewidth=2, label='Sun direction')

ax.set_xlabel('X (m)')
ax.set_ylabel('Y (m)')
ax.set_zlabel('Z (m)')
ax.legend(loc='upper left', fontsize=7)

def update(k):
    global sun_arrow
    i = frame_idx[k]

    # Update spacecraft markers
    oculus_pt.set_data([sim_year.r1[i, 0]], [sim_year.r1[i, 1]])
    oculus_pt.set_3d_properties([sim_year.r1[i, 2]])
    lux_pt.set_data([sim_year.r2[i, 0]], [sim_year.r2[i, 1]])
    lux_pt.set_3d_properties([sim_year.r2[i, 2]])

    # Update Sun arrow
    sun_arrow.remove()
    sun_dir = sun_rel[i]
    sun_hat = sun_dir / np.linalg.norm(sun_dir)
    base    = sun_hat * gc_scale
    tip     = sun_hat * (gc_scale * 0.15)
    sun_arrow = ax.quiver(
        base[0], base[1], base[2],
        tip[0] - base[0], tip[1] - base[1], tip[2] - base[2],
        color='orange', linewidth=2
    )

    # --- Camera: always perpendicular to Sun direction and Earth North Pole ---
    # North pole of Earth in ECI is Z-axis
    north = np.array([0, 0, 1])
    sun_hat_xy = sun_hat.copy()

    # Right vector: cross(north, sun_hat) — perpendicular to both
    right = np.cross(north, sun_hat)
    right /= np.linalg.norm(right)

    # Camera sits along the 'right' axis, elevated slightly
    cam_dir = right  # view from the side perpendicular to Sun

    # Convert to azimuth/elevation for matplotlib
    azim = np.degrees(np.arctan2(cam_dir[1], cam_dir[0]))
    elev = np.degrees(np.arcsin(np.clip(cam_dir[2], -1, 1)))

    ax.view_init(elev=15, azim=azim)

    return oculus_pt, lux_pt, sun_arrow

anim = FuncAnimation(fig_gif, update, frames=n_frames, interval=80, blit=False)
anim.save('orbits_rotating_camera.gif', writer=PillowWriter(fps=15))
plt.close(fig_gif)
print('Saved orbits_rotating_camera.gif')

Saved orbits_rotating_camera.gif


In [ ]:
import plotly.graph_objects as go
from plotly.io import to_html

figures = [
    ("Sun-centered static view — full year trajectories", fig_sun_centered),
    ("Earth-centered static view — one orbit", fig_static),
    ("Full year animation — Sun direction & spacecraft", fig_animated)
]

# Convert each figure to an HTML div (no full page, no extra plotly.js)
divs = []
for i, (title, fig) in enumerate(figures):
    include_js = (i == 0)
    div = to_html(
        fig,
        full_html=False,
        include_plotlyjs='cdn' if include_js else False,
        div_id=f'plot_{i}'
    )
    divs.append((title, div))

# Build the full HTML page
html_parts = ["""<!DOCTYPE html>
<html lang="en">
<head>
<meta charset="UTF-8">
<meta name="viewport" content="width=device-width, initial-scale=1.0">
<title>XEUS Formation Flying — Mission Analysis</title>
<script src="https://cdn.plot.ly/plotly-latest.min.js"></script>
<style>
  body {
    font-family: 'Segoe UI', Arial, sans-serif;
    background: #0d0d1a;
    color: #e0e0f0;
    margin: 0;
    padding: 0;
  }
  header {
    background: #13132b;
    padding: 2rem 3rem 1.5rem;
    border-bottom: 1px solid #2a2a5a;
  }
  header h1 {
    margin: 0 0 0.4rem;
    font-size: 1.8rem;
    color: #a0c4ff;
    letter-spacing: 0.05em;
  }
  header p {
    margin: 0;
    color: #7080a0;
    font-size: 0.95rem;
  }
  .section {
    margin: 2.5rem 3rem;
  }
  .section h2 {
    font-size: 1.1rem;
    color: #7eb8f7;
    margin-bottom: 0.6rem;
    border-left: 3px solid #3a6fcc;
    padding-left: 0.75rem;
    letter-spacing: 0.03em;
  }
  .plot-container {
    background: #12122a;
    border: 1px solid #1e2050;
    border-radius: 8px;
    overflow: hidden;
    box-shadow: 0 4px 24px rgba(0,0,0,0.5);
  }
  footer {
    text-align: center;
    padding: 2rem;
    color: #3a4060;
    font-size: 0.8rem;
    border-top: 1px solid #1a1a3a;
    margin-top: 3rem;
  }
</style>
</head>
<body>
<header>
  <h1>XEUS Formation Flying — Mission Analysis</h1>
  <p>Orbital dynamics, Galactic Center alignment, and formation geometry</p>
</header>
"""]

for title, div in divs:
    html_parts.append(f"""
<div class="section">
  <h2>{title}</h2>
  <div class="plot-container">
    {div}
  </div>
</div>
""")

html_parts.append("""
<footer>Generated with Plotly &mdash; XEUS Mission GNC Analysis</footer>
</body>
</html>
""")

full_html = "\n".join(html_parts)

with open('xeus_mission_analysis.html', 'w', encoding='utf-8') as f:
    f.write(full_html)

print('Saved xeus_mission_analysis.html')

Saved xeus_mission_analysis.html


In [16]:
import numpy as np
import astropy.units as u
from astropy.time import Time
from astropy.coordinates import SkyCoord, Galactic, GeocentricMeanEcliptic
import plotly.graph_objects as go


# ============================================================
# Helpers
# ============================================================

def unit(v):
    v = np.asarray(v, dtype=float)
    return v / np.linalg.norm(v)


def orthonormal_basis_from_normal(nhat):
    nhat = unit(nhat)
    if abs(nhat[2]) < 0.9:
        ref = np.array([0.0, 0.0, 1.0])
    else:
        ref = np.array([1.0, 0.0, 0.0])

    e1 = unit(np.cross(nhat, ref))
    e2 = unit(np.cross(nhat, e1))
    return e1, e2


def make_circular_disk(center, normal_hat, radius, nr=40, nt=120):
    e1, e2 = orthonormal_basis_from_normal(normal_hat)
    r = np.linspace(0.0, radius, nr)
    th = np.linspace(0.0, 2*np.pi, nt)
    R, TH = np.meshgrid(r, th)

    X = center[0] + R*np.cos(TH)*e1[0] + R*np.sin(TH)*e2[0]
    Y = center[1] + R*np.cos(TH)*e1[1] + R*np.sin(TH)*e2[1]
    Z = center[2] + R*np.cos(TH)*e1[2] + R*np.sin(TH)*e2[2]
    return X, Y, Z


def make_arc_between_vectors(v1, v2, radius, n=120):
    v1 = unit(v1)
    v2 = unit(v2)

    axis = np.cross(v1, v2)
    axis_norm = np.linalg.norm(axis)
    if axis_norm < 1e-12:
        pts = np.outer(np.linspace(0, 1, n), v1) * radius
        return pts[:, 0], pts[:, 1], pts[:, 2]

    axis = axis / axis_norm
    angle = np.arccos(np.clip(np.dot(v1, v2), -1.0, 1.0))
    tvals = np.linspace(0.0, angle, n)

    pts = []
    for t in tvals:
        vrot = (
            v1*np.cos(t)
            + np.cross(axis, v1)*np.sin(t)
            + axis*np.dot(axis, v1)*(1 - np.cos(t))
        )
        pts.append(radius * vrot)

    pts = np.array(pts)
    return pts[:, 0], pts[:, 1], pts[:, 2]


def add_arrow(fig, start, end, color, name, width=6, cone_scale=0.12, showlegend=True):
    start = np.asarray(start, dtype=float)
    end = np.asarray(end, dtype=float)

    vec = end - start
    L = np.linalg.norm(vec)
    if L == 0:
        return

    vhat = vec / L
    shaft_end = end - cone_scale * L * vhat

    fig.add_trace(go.Scatter3d(
        x=[start[0], shaft_end[0]],
        y=[start[1], shaft_end[1]],
        z=[start[2], shaft_end[2]],
        mode="lines",
        name=name,
        showlegend=showlegend,
        line=dict(color=color, width=width),
        hovertemplate=f"{name}<extra></extra>"
    ))

    fig.add_trace(go.Cone(
        x=[end[0]],
        y=[end[1]],
        z=[end[2]],
        u=[cone_scale * L * vhat[0]],
        v=[cone_scale * L * vhat[1]],
        w=[cone_scale * L * vhat[2]],
        anchor="tip",
        sizemode="absolute",
        sizeref=0.18 * L,
        colorscale=[[0, color], [1, color]],
        showscale=False,
        showlegend=False,
        hoverinfo="skip"
    ))


# ============================================================
# Units
# ============================================================

pc = 3.085677581491367e16
kpc = 1e3 * pc

def to_kpc(vec_m):
    return np.asarray(vec_m) / kpc


# ============================================================
# Galactic geometry
# ============================================================

t0epoch = Time("2000-01-01T00:00:00", scale="utc")

R0_m = 8.2 * kpc
sun_gal_m = np.array([R0_m, 0.0, 0.0])

ngp_hat_gal = np.array([0.0, 0.0, 1.0])

nep_gal_coord = SkyCoord(
    lon=0*u.deg,
    lat=90*u.deg,
    frame=GeocentricMeanEcliptic(equinox=t0epoch)
).transform_to(Galactic)

nep_hat_gal = unit(np.array([
    nep_gal_coord.cartesian.x.to_value(u.one),
    nep_gal_coord.cartesian.y.to_value(u.one),
    nep_gal_coord.cartesian.z.to_value(u.one),
]))

gcsun_hat = unit(sun_gal_m)

tilt_plane_deg = np.degrees(
    np.arccos(np.clip(np.dot(ngp_hat_gal, nep_hat_gal), -1.0, 1.0))
)

print(f"Galactic/ecliptic plane tilt = {tilt_plane_deg:.3f} deg")


# ============================================================
# Figure geometry
# ============================================================

gal_plane_radius_m = 3.2 * kpc
ecl_disk_radius_m = 1.0 * kpc  # exaggerated for visibility

Xg, Yg, Zg = make_circular_disk(
    center=np.zeros(3),
    normal_hat=ngp_hat_gal,
    radius=gal_plane_radius_m,
    nr=35,
    nt=120
)

Xe, Ye, Ze = make_circular_disk(
    center=sun_gal_m,
    normal_hat=nep_hat_gal,
    radius=ecl_disk_radius_m,
    nr=35,
    nt=120
)


# ============================================================
# Angle marker: pole-to-pole only
# ============================================================

pole_arc_radius_m = 1.2 * kpc
Xp, Yp, Zp = make_arc_between_vectors(ngp_hat_gal, nep_hat_gal, pole_arc_radius_m, n=140)

pole_arc_shift = np.array([0.18 * kpc, 0.15 * kpc, 0.10 * kpc])
Xp += pole_arc_shift[0]
Yp += pole_arc_shift[1]
Zp += pole_arc_shift[2]

pole_label_pos = to_kpc(pole_arc_shift + 1.35 * kpc * unit(ngp_hat_gal + nep_hat_gal))


# ============================================================
# Build figure
# ============================================================

fig = go.Figure()

fig.add_trace(go.Surface(
    x=to_kpc(Xg), y=to_kpc(Yg), z=to_kpc(Zg),
    name="Galactic plane",
    showscale=False,
    opacity=0.25,
    colorscale=[[0, "royalblue"], [1, "royalblue"]],
    hovertemplate="Galactic plane<extra></extra>"
))

fig.add_trace(go.Surface(
    x=to_kpc(Xe), y=to_kpc(Ye), z=to_kpc(Ze),
    name="Ecliptic plane",
    showscale=False,
    opacity=0.60,
    colorscale=[[0, "limegreen"], [1, "limegreen"]],
    hovertemplate="Ecliptic plane<extra></extra>"
))

fig.add_trace(go.Scatter3d(
    x=[0], y=[0], z=[0],
    mode="markers",
    name="Galactic Center",
    marker=dict(size=18, color="red"),
    hovertemplate="Galactic Center<extra></extra>"
))

fig.add_trace(go.Scatter3d(
    x=[to_kpc(sun_gal_m)[0]],
    y=[to_kpc(sun_gal_m)[1]],
    z=[to_kpc(sun_gal_m)[2]],
    mode="markers",
    name="Sun",
    marker=dict(size=9, color="orange"),
    hovertemplate="Sun<extra></extra>"
))

fig.add_trace(go.Scatter3d(
    x=[0, to_kpc(sun_gal_m)[0]],
    y=[0, to_kpc(sun_gal_m)[1]],
    z=[0, to_kpc(sun_gal_m)[2]],
    mode="lines",
    name="GC-Sun line",
    line=dict(color="orange", width=5, dash="dash"),
    hovertemplate="GC-Sun line<extra></extra>"
))

add_arrow(
    fig,
    [0, 0, 0],
    to_kpc(2.0 * kpc * ngp_hat_gal),
    color="deepskyblue",
    name="North Galactic Pole",
    width=7
)

add_arrow(
    fig,
    to_kpc(sun_gal_m),
    to_kpc(sun_gal_m + 1.4 * kpc * nep_hat_gal),
    color="green",
    name="North Ecliptic Pole",
    width=7
)

fig.add_trace(go.Scatter3d(
    x=to_kpc(Xp),
    y=to_kpc(Yp),
    z=to_kpc(Zp),
    mode="lines",
    name="Pole tilt",
    line=dict(color="black", width=7),
    hovertemplate=f"NGP–NEP angle = {tilt_plane_deg:.2f} deg<extra></extra>"
))

fig.add_trace(go.Scatter3d(
    x=[pole_label_pos[0]],
    y=[pole_label_pos[1]],
    z=[pole_label_pos[2]],
    mode="text",
    text=[f"NGP–NEP<br>{tilt_plane_deg:.1f}°"],
    textfont=dict(size=18, color="black"),
    showlegend=False,
    hoverinfo="skip"
))


# ============================================================
# Camera presets
# ============================================================

camera_presets = {
    "isometric": dict(eye=dict(x=1.45, y=1.30, z=0.95)),
    "topdown":   dict(eye=dict(x=0.05, y=0.05, z=2.35)),
    "edgeon":    dict(eye=dict(x=0.05, y=2.40, z=0.05)),
    "sun_side":  dict(eye=dict(x=2.20, y=0.55, z=0.35)),
    "gc_to_sun": dict(eye=dict(x=2.35, y=0.02, z=0.18)),
}

view_name = "isometric"
camera = camera_presets[view_name]


# ============================================================
# Layout
# ============================================================

fig.update_layout(
    width=1100,
    height=700,
    font=dict(size=20, family="Arial"),
    legend=dict(font=dict(size=18)),
    margin=dict(l=10, r=10, b=10, t=60),
    scene=dict(
        xaxis=dict(
            title=dict(text="Galactic X (kpc)", font=dict(size=20)),
            tickfont=dict(size=16),
        ),
        yaxis=dict(
            title=dict(text="Galactic Y (kpc)", font=dict(size=20)),
            tickfont=dict(size=16),
        ),
        zaxis=dict(
            title=dict(text="Galactic Z (kpc)", font=dict(size=20)),
            tickfont=dict(size=16),
        ),
        aspectmode="manual",
        aspectratio=dict(x=1.8, y=1.0, z=1.0),
        camera=camera,
    ),
)

fig.write_image(
    "galactic_frame_static.pdf",
    width=1100,
    height=700,
)

Galactic/ecliptic plane tilt = 60.194 deg


In [17]:
import plotly.graph_objects as go
from plotly.io import to_html

figures = [
    ("Galactic-centered view of the solar system", fig),
    ("Sun centered view of the formation", fig_sun_centered),
    ("Full year animation — Sun direction & spacecraft", fig_animated)
]

# Convert each figure to an HTML div (no full page, no extra plotly.js)
divs = []
for i, (title, fig) in enumerate(figures):
    include_js = (i == 0)
    div = to_html(
        fig,
        full_html=False,
        include_plotlyjs='cdn' if include_js else False,
        div_id=f'plot_{i}'
    )
    divs.append((title, div))

# Build the full HTML page
html_parts = ["""<!DOCTYPE html>
<html lang="en">
<head>
<meta charset="UTF-8">
<meta name="viewport" content="width=device-width, initial-scale=1.0">
<title>XEUS Formation Flying — Mission Analysis</title>
<script src="https://cdn.plot.ly/plotly-latest.min.js"></script>
<style>
  body {
    font-family: 'Segoe UI', Arial, sans-serif;
    background: #0d0d1a;
    color: #e0e0f0;
    margin: 0;
    padding: 0;
  }
  header {
    background: #13132b;
    padding: 2rem 3rem 1.5rem;
    border-bottom: 1px solid #2a2a5a;
  }
  header h1 {
    margin: 0 0 0.4rem;
    font-size: 1.8rem;
    color: #a0c4ff;
    letter-spacing: 0.05em;
  }
  header p {
    margin: 0;
    color: #7080a0;
    font-size: 0.95rem;
  }
  .section {
    margin: 2.5rem 3rem;
  }
  .section h2 {
    font-size: 1.1rem;
    color: #7eb8f7;
    margin-bottom: 0.6rem;
    border-left: 3px solid #3a6fcc;
    padding-left: 0.75rem;
    letter-spacing: 0.03em;
  }
  .plot-container {
    background: #12122a;
    border: 1px solid #1e2050;
    border-radius: 8px;
    overflow: hidden;
    box-shadow: 0 4px 24px rgba(0,0,0,0.5);
  }
  footer {
    text-align: center;
    padding: 2rem;
    color: #3a4060;
    font-size: 0.8rem;
    border-top: 1px solid #1a1a3a;
    margin-top: 3rem;
  }
</style>
</head>
<body>
<header>
  <h1>XEUS Formation Flying — Mission Analysis</h1>
  <p>Orbital dynamics, Galactic Center alignment, and formation geometry</p>
</header>
"""]

for title, div in divs:
    html_parts.append(f"""
<div class="section">
  <h2>{title}</h2>
  <div class="plot-container">
    {div}
  </div>
</div>
""")

html_parts.append("""
<footer>Generated with Plotly &mdash; XEUS Mission GNC Analysis</footer>
</body>
</html>
""")

full_html = "\n".join(html_parts)

with open('xeus_mission_analysis.html', 'w', encoding='utf-8') as f:
    f.write(full_html)

print('Saved xeus_mission_analysis.html')

Saved xeus_mission_analysis.html


In [13]:
import plotly.graph_objects as go
import numpy as np

# ── Precompute per-frame data from existing sim_year ──────────────────────────
sun_geo   = np.array([r_sun_at(ti) for ti in t])   # Sun position in ECI at each step
r_oc      = sim_year.r1                              # Oculus positions
r_lx      = sim_year.r2                              # Lux positions

arrow_len = R_orbit * 0.4
TRAIL     = 30  # number of steps to show as orbit trail

def _unit(v):
    return v / np.linalg.norm(v)

# ── Build frames ──────────────────────────────────────────────────────────────
frames = []
for fi in range(len(sun_geo)):
    oc  = r_oc[fi]
    lx  = r_lx[fi]
    mid = (oc + lx) / 2.0

    sun_unit   =  _unit(sun_geo[fi])        # toward Sun
    earth_unit = -_unit(r_oc[fi])           # toward Earth (from spacecraft)

    def arrow_traces(origin, unit, color, name):
        tip       = origin + unit * arrow_len
        shaft_end = origin + unit * arrow_len * 0.85
        shaft = go.Scatter3d(
            x=[origin[0], shaft_end[0]],
            y=[origin[1], shaft_end[1]],
            z=[origin[2], shaft_end[2]],
            mode='lines', line=dict(color=color, width=5),
            name=name, showlegend=True
        )
        cone = go.Cone(
            x=[tip[0]], y=[tip[1]], z=[tip[2]],
            u=[unit[0]*arrow_len*0.15],
            v=[unit[1]*arrow_len*0.15],
            w=[unit[2]*arrow_len*0.15],
            colorscale=[[0, color], [1, color]],
            showscale=False, sizemode='absolute',
            sizeref=R_orbit * 0.04,
            showlegend=False
        )
        return shaft, cone

    sun_shaft,   sun_cone   = arrow_traces(mid, sun_unit,   'yellow',     'To Sun')
    earth_shaft, earth_cone = arrow_traces(mid, earth_unit, 'deepskyblue','To Earth')

    trail_start = max(0, fi - TRAIL)
    day = fi / N_steps * 365.25

    frame_data = [
        # Spacecraft markers
        go.Scatter3d(x=[oc[0]], y=[oc[1]], z=[oc[2]],
                     mode='markers', marker=dict(size=6, color='white', symbol='square'),
                     name='Oculus'),
        go.Scatter3d(x=[lx[0]], y=[lx[1]], z=[lx[2]],
                     mode='markers', marker=dict(size=6, color='lightblue', symbol='square'),
                     name='Lux'),
        # Orbit trails
        go.Scatter3d(x=r_oc[trail_start:fi+1, 0],
                     y=r_oc[trail_start:fi+1, 1],
                     z=r_oc[trail_start:fi+1, 2],
                     mode='lines', line=dict(color='white', width=1),
                     name='Oculus trail', showlegend=False),
        go.Scatter3d(x=r_lx[trail_start:fi+1, 0],
                     y=r_lx[trail_start:fi+1, 1],
                     z=r_lx[trail_start:fi+1, 2],
                     mode='lines', line=dict(color='lightblue', width=1),
                     name='Lux trail', showlegend=False),
        # Earth marker
        go.Scatter3d(x=[0], y=[0], z=[0],
                     mode='markers', marker=dict(size=12, color='dodgerblue'),
                     name='Earth'),
        # Arrows
        sun_shaft, sun_cone,
        earth_shaft, earth_cone,
    ]

    frames.append(go.Frame(
        data=frame_data, name=str(fi),
        layout=go.Layout(title_text=f"Day {day:.1f} / 365.25")
    ))

# ── Figure ────────────────────────────────────────────────────────────────────
fig = go.Figure(data=frames[0].data, frames=frames)

ax_range = R_orbit * 1.3
fig.update_layout(
    title="X-Ray Telescope Formation — Sun & Earth Direction Vectors (1 Year)",
    paper_bgcolor='#0d0d0d',
    font=dict(color='white'),
    scene=dict(
        xaxis=dict(title='X (m)', range=[-ax_range, ax_range],
                   backgroundcolor='#0d0d0d', gridcolor='#333', color='white'),
        yaxis=dict(title='Y (m)', range=[-ax_range, ax_range],
                   backgroundcolor='#0d0d0d', gridcolor='#333', color='white'),
        zaxis=dict(title='Z (m)', range=[-ax_range, ax_range],
                   backgroundcolor='#0d0d0d', gridcolor='#333', color='white'),
        aspectmode='cube', bgcolor='#0d0d0d',
    ),
    updatemenus=[dict(
        type='buttons', showactive=False,
        y=0.02, x=0.5, xanchor='center',
        buttons=[
            dict(label='▶ Play', method='animate',
                 args=[None, dict(frame=dict(duration=50, redraw=True),
                                  fromcurrent=True, mode='immediate')]),
            dict(label='⏸ Pause', method='animate',
                 args=[[None], dict(frame=dict(duration=0, redraw=False),
                                    mode='immediate')])
        ]
    )],
    sliders=[dict(
        steps=[dict(method='animate',
                    args=[[str(k)], dict(mode='immediate',
                                         frame=dict(duration=0, redraw=True))],
                    label=f"{k/N_steps*365:.0f}d") for k in range(N_steps)],
        transition=dict(duration=0),
        x=0.05, y=0, len=0.9,
        currentvalue=dict(prefix='Day: ', font=dict(color='white')),
        font=dict(color='white')
    )],
    margin=dict(l=0, r=0, b=80, t=50),
)

fig.write_html("formation_flight_viz.html", include_plotlyjs='cdn')
print("Saved: formation_flight_viz.html")

Saved: formation_flight_viz.html
